# <div style="text-align: center"> Group project: Does the weather affect sales? 🌦️🛒

### <span style="color:red"> GROUP NUMBER: ____ &nbsp;&nbsp; CARD: ____ &nbsp;&nbsp; MEMBERS: ______________________________ </span>

**Before you start:** save a copy of this notebook as `groupN_sales_weather.ipynb` (N = your group number).

## 🎯 The mission
The marketing director of our online gift shop (the same Online Retail data, Dec 2010 - Dec 2011) asks:

> ***"Should we plan our email campaigns and our stock around the weather forecast?"***

Each group looks at the question from **one angle** and builds **one class** that answers it. All classes inherit from `Analysis`
(in `sales_weather_base.py`) and work like scikit-learn models: **create → `fit()` → use**. Another group will use your class - so make it work,
test it, and document it so they can use it without asking you.

### ⏱️ Timeline (70 min + presentations)
| Time | Step |
|---|---|
| 0-5 min | Read your card, split the roles (everybody codes!) |
| 5-15 | **Spec**: write the class line, the method signatures + docstrings, and your tests - *before* any body |
| 15-45 | **Build**: AI writes bodies, tests decide. Answer your card's business question |
| 45-55 | **Swap test**: send your `.py` file to the next group (group 1 → 2, ..., last → 1). They use it with `help()` only |
| 55-70 | **Prepare the presentation** (the 3 cells right below) + finish the AI log |
| then | 🎲 **Random group, random presenter**: 5 min + 1 question about your code. *Everybody must be able to present.* |

## ✅ Definition of done = how you are graded (0-2 points each, max 8)

| Criterion | 0 | 1 | 2 |
|---|---|---|---|
| **Works** | doesn't run | runs, but wrong/unchecked numbers | runs on the real data and the numbers make sense (you checked!) |
| **Tested & safe** | no tests | some asserts | ≥ 2 asserts per method **+** bad input raises a clear error (`ValueError`, or `RuntimeError` before `fit()`) |
| **Reusable** | code only in the notebook | `.py` module, weak docstrings | `.py` module with a class and docstrings good enough that the swap group could use it |
| **Explained** | presenter can't explain | explains *what*, not *how* | presenter explains any line I point at + honest AI log + **one reason the number might not be real** |

**Submission (today, 23:59, Blackboard):** `groupN_sales_weather.ipynb` **and** `groupN_<module>.py` - one upload per group, all names in the notebook.

---
# 🎤 PRESENTATION (fill in at the end - this is what you show on the screen)

### 1️⃣ What we built
*Class name and its methods (one line each: what goes in, what comes out). Show `fit()` and one method live.*

- `MyClass(data)` → ...
- `.fit()` → ...
- `.method(...)` → ...

### 2️⃣ Our answer to the marketing director
*Your card's question, your answer in 1-2 sentences, and the number/table/chart that proves it (the code cell is in "Step 4" below).*

### 3️⃣ Is our number real? + what the AI got wrong
*One reason your result might be misleading (season, weekday, missing days, small sample, a threshold you chose...),
and one concrete moment when the AI's code was wrong or misleading.*

---
# ⚙️ Setup (run it, don't change it)

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sales_weather_base import (load_sales_weather, load_daily_sales, load_weather,
                                load_transactions, Analysis, WEEKDAYS, expect_error)

data = load_sales_weather()      # one row per day WITH sales: sales + weather (first run: a few seconds)
print(data.shape)
data.head()

### 📖 Data dictionary - `data` (one row = one day with sales)
| Column | Meaning |
|---|---|
| *index* `date` | the day (a real date - you can use `data.index.month`, `data.index.day_name()`, `data.loc["2011-07"]`) |
| `revenue` | sales of the day in GBP (cleaned with `retail_tools.cleaning.clean_retail` + cancelled orders removed) |
| `orders`, `customers`, `units` | number of invoices, number of customers **with a CustomerID** (guest orders have none), number of pieces sold that day |
| `temp_mean`, `temp_max`, `temp_min` | London temperature, °C |
| `rain_mm` | precipitation (rain + melted snow), mm |
| `snow_cm`, `sunshine_h`, `wind_kmh` | snowfall (cm), hours of sunshine, max wind speed (km/h) |
| `weekday`, `month` | e.g. `"Monday"`, `7` |

⚠️ **Days without any sales are not in `data`** - you saw it in Task 7 (305 of 374 days).
⚠️ The customers are mostly **shops (wholesalers)**, not people buying an umbrella on the way home.

### 🧬 Your class inherits from `Analysis` - what you get for free
```python
class MyAnalysis(Analysis):
    required_columns = ["revenue", "rain_mm"]   # Analysis checks that data has these columns
```
| You get | What it does |
|---|---|
| `MyAnalysis(data)` | checks `data` is a DataFrame (`TypeError`) with the `required_columns` (`ValueError`), stores a **copy** in `self.data`, sets `self.results_ = None` |
| `self.check_fitted()` | raises `RuntimeError` if `fit()` hasn't been called - call it at the start of every method except `fit()` |
| `print(obj)` | `MyAnalysis(305 rows, fitted)` |

**You write:** `fit()` (compute → store in `self.results_` → `return self`) and the methods on your card.
If your class needs **extra arguments** (Cards 2 and 5), write your own `__init__` and call `super().__init__(data)` first - like `DiscountBasket`.

In [ ]:
help(Analysis)

---
# 🃏 The cards
Each group gets **one card**. (If there are more groups than cards, two groups get the same card - and we compare!)
Class names, method names and arguments are **requirements** - the swap group will call them exactly like this.

## Card 1 - `missing_days.py` - the calendar team 📅
Works on **`daily = load_daily_sales()`** (sales only). **Don't** call your file `calendar.py` - why would that break things? (Remember `time.py` last week.)

| Method | Does / returns |
|---|---|
| `MissingDays(daily)` | `required_columns = ["revenue"]` |
| `.fit()` | finds every calendar day between the first and the last day of `daily` that has **no** sales; stores them in `self.results_`; returns `self` |
| `.missing_days()` | **sorted list** of the missing days as `"YYYY-MM-DD"` strings |
| `.by_weekday()` | **dict** `{weekday: number of missing days}`, Monday first |
| `.save_report(path)` | writes a **JSON file** `{"n_missing": ..., "by_weekday": {...}, "not_saturday": [dates]}` and returns `path`. Test it by reading the file back with `json.load`! |

**Business question for your presentation:** Sales exist on only 305 of 374 days. **Which days are missing - data errors or real closures?** Explain every missing day that is not a Saturday. What does it mean for the averages the other groups compute?

*Useful:* `pd.date_range(start, end, freq="D")`, `.difference()`, `.day_name()`, `.strftime("%Y-%m-%d")`

## Card 2 - `rain.py` - does rain sell? 🌧️


| Method | Does / returns |
|---|---|
| `RainEffect(data, threshold=1.0)` | `required_columns = ["revenue", "rain_mm"]`. Own `__init__`: call `super().__init__(data)` first, then store `threshold`; negative threshold → `ValueError` |
| `.fit()` | a day is rainy if `rain_mm >= threshold`; `self.results_` = **dict** `{"rainy_days", "dry_days", "rainy_mean", "dry_mean", "difference_pct"}` (means = mean revenue; `difference_pct` = how many % higher the rainy mean is); returns `self` |
| `.by_month()` | **DataFrame**, one row per month (1-12), columns `"dry"` and `"rainy"` = mean revenue |

**Business question for your presentation:** Should marketing send campaigns on rainy days? **Compare threshold 1 mm and 5 mm.** Does your answer survive? Is it the rain - or the season (look at `by_month()`)?

*Useful:* a True/False Series `rainy` → `.loc[rainy, "revenue"]` and `.loc[~rainy, "revenue"]`

## Card 3 - `temperature.py` - hot or cold? 🌡️


| Method | Does / returns |
|---|---|
| `TemperatureEffect(data)` | `required_columns = ["revenue", "temp_mean"]` |
| `.fit()` | `self.results_` = **dict** `{"overall": r, "by_month": {1: r, 2: r, ...}, "days_per_month": {1: n, ...}}` where r = correlation of `revenue` and `temp_mean`, rounded to 2 decimals; returns `self` |
| `.strongest_month()` | the month (1-12) with the largest **absolute** correlation |
| `.plot()` | scatter plot temperature vs revenue; returns the matplotlib `Axes` |

**Business question for your presentation:** Do warm or cold days sell more? Is the month-by-month picture consistent? **How many days** is each monthly correlation based on - would you trust it?

*Useful:* `series_a.corr(series_b)`; group by `data.index.month`

## Card 4 - `weekdays.py` - the weekday rhythm 🗓️


| Method | Does / returns |
|---|---|
| `WeekdayEffect(data)` | `required_columns = ["revenue"]` |
| `.fit()` | `self.results_` = **dict** `{weekday: mean revenue}` in the order Monday → Sunday, **only weekdays that have data**; returns `self` |
| `.best_day()` | the weekday with the highest mean revenue |
| `.ratio(day_a, day_b)` | how many times higher the mean of `day_a` is than `day_b` (1 decimal); an unknown name **or** a day without sales → `ValueError` |
| `.plot()` | bar chart of `results_`; returns the `Axes` |

**Business question for your presentation:** How big is the weekday effect **compared with the weather effects** the other groups find? On which day would you send a campaign email - and which day can't you analyse at all?

*Useful:* `WEEKDAYS` (imported in the setup) gives the right order

## Card 5 - `products_weather.py` - weather products ☂️🧣
Works on **transactions**, not on daily totals: `tr = load_transactions()` and `weather = load_weather()`.

| Method | Does / returns |
|---|---|
| `ProductWeather(transactions, weather, keyword)` | `required_columns = ["Description", "Quantity", "InvoiceDate"]`. Own `__init__`: `super().__init__(transactions)`, then store `weather` and `keyword` (upper case); empty keyword → `ValueError` |
| `.fit()` | **weekly** units sold of all products whose `Description` contains the keyword + **weekly** mean temperature; `self.results_` = **DataFrame** with columns `units`, `temp_mean`; no matching product → `ValueError`; returns `self` |
| `.correlation()` | correlation of `units` and `temp_mean`, 2 decimals |
| `.plot()` | units and temperature over time on one chart (second y-axis: `ax.twinx()`); returns the `Axes` |

**Business question for your presentation:** Which products are "weather products"? Try `HOT WATER BOTTLE`, `PARASOL`, `PICNIC`, `LUNCH BAG` and two of your own. **Is a high correlation proof that the weather drives them?** Look at the plot, not just r.

*Useful:* `.str.contains(keyword, regex=False)`, `.set_index("InvoiceDate")["Quantity"].resample("W").sum()`

## Card 6 - `forecast.py` - can we forecast sales? 🔮
Split by time: `train = data.loc[:"2011-09-30"]`, `test = data.loc["2011-10-01":]`. Fit on `train`, score on `test`.

| Method | Does / returns |
|---|---|
| `WeekdayForecaster(train)` | `required_columns = ["revenue"]` |
| `.fit()` | learns the mean revenue **per weekday** (store in `self.results_`) and the overall mean (`self.overall_mean_`); returns `self` |
| `.predict(dates)` | **NumPy array**: for each date the mean of its weekday; a weekday not seen in training → the overall mean; not fitted → `RuntimeError` |
| `.score(test)` | **mean absolute error** (MAE) in GBP on a DataFrame with a date index and `revenue` |

**Business question for your presentation:** Does your forecaster beat the **baseline** (always predict the training mean - Part 2's `MeanPredictor`)? By how much? Would adding rain help? Try a **random** train/test split instead - why does it look better, and why is it cheating?

*Useful:* `pd.to_datetime(dates).day_name()`; `dict.get(key, default)`; MAE = `np.mean(np.abs(actual - predicted))`

---
# 🛠️ Work area

## Step 1 - Spec: your module with the class, signatures and docstrings
Rename the file (`groupN_<module>.py`, e.g. `group2_rain.py`), write the class line, `required_columns` and all method
signatures + docstrings, leave the bodies as `...`. Run the cell to save the file.
Then (Step 3) write your tests. **Only then** fill in the bodies - with Copilot - and re-run this cell after every change.

In [ ]:
%%writefile group0_mymodule.py
"""Group 0 - <module>: <one sentence: what this module does>.

Authors: <names>
"""
import numpy as np
import pandas as pd

from sales_weather_base import Analysis, WEEKDAYS


class MyAnalysis(Analysis):    # <- rename to your card's class, e.g. RainEffect
    """<What this class analyses, in one sentence>.

    Usage:
        model = MyAnalysis(data).fit()
        model.my_method()
    """
    required_columns = ["revenue"]

    def fit(self):
        """<What is computed>. Stores it in self.results_ and returns self."""
        ...
        return self

    def my_method(self):
        """<What it returns>.

        Raises:
            RuntimeError: if fit() was not called
        """
        self.check_fitted()
        ...

## Step 2 - Import and try it

In [ ]:
import group0_mymodule as gm      # <- change to your module name

help(gm)

In [ ]:
# try your class on the real data here

## Step 3 - Tests
At least **2 asserts per method** + checks that bad input / a missing `fit()` raise errors. Some ideas:
```python
model = gm.MyAnalysis(data)
assert model.fit() is model                                  # fit() returns self (Task 3!)
assert isinstance(model.results_, dict)                      # the right type
assert expect_error(gm.MyAnalysis(data).my_method, error=RuntimeError)   # not fitted yet
assert expect_error(gm.MyAnalysis, "not a table", error=TypeError)       # comes from Analysis
```
**Check at least one number by hand** (filter `data` yourself and compare).

In [ ]:
# tests

print("All tests passed ✅")

## Step 4 - Answer the business question (use YOUR class)

In [ ]:
# business question

## Step 5 - Swap test 🔄
Import the module you received from group ____ . Use **only** `help()` - don't ask its authors. Then write below:
- Did it work on the first try? ____
- Was the documentation enough? ____
- One bug or one suggestion: ____

**Send these notes back to the authors** - they may fix it before submitting.

In [ ]:
# import groupX_module as other
# help(other)

---
## 🤖 AI log (required - this is your disclosure)
| # | Tool | Prompt (what we asked) | What was wrong / what we changed |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

## 👥 Who did what
| Name | Contribution |
|---|---|
| | |
| | |
| | |
| | |